In [ ]:
#ADDB Periodic Config 110-170 bpm
# vanilla 

In [ ]:
# ========================================================================
# PERFECG-Net: UNet1D with Integrated Vanilla Transformer Attention
# ========================================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ========================================================================
# Vanilla Transformer Attention Block
# ========================================================================
class SparseAttentionBlock(nn.Module):
    """
    Vanilla transformer attention block for fetal ECG.
    Uses standard MultiheadAttention from PyTorch.
    Handles projections and reshaping to fit into the U-Net architecture.
    """
    def __init__(self, in_channels, nhead, fs, bpm_range, dropout=0.1):
        super().__init__()
        self.nhead = nhead
        self.d_model = in_channels
        self.d_head = self.d_model // nhead

        # Linear projections for Query, Key, Value
        self.query_projection = nn.Conv1d(in_channels, self.d_model, kernel_size=1)
        self.key_projection = nn.Conv1d(in_channels, self.d_model, kernel_size=1)
        self.value_projection = nn.Conv1d(in_channels, self.d_model, kernel_size=1)

        # Vanilla Transformer Multihead Attention
        self.attention = nn.MultiheadAttention(
            embed_dim=self.d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )

        # Final output projection
        self.out_projection = nn.Conv1d(self.d_model, in_channels, kernel_size=1)

    def forward(self, x):
        """ Input x shape: (B, C, T) """
        B, C, T = x.shape

        # Project to query, key, value
        q = self.query_projection(x)  # (B, C, T)
        k = self.key_projection(x)    # (B, C, T)
        v = self.value_projection(x)  # (B, C, T)

        # Reshape to (B, T, C) for MultiheadAttention (batch_first=True)
        q = q.transpose(1, 2)  # (B, T, C)
        k = k.transpose(1, 2)  # (B, T, C)
        v = v.transpose(1, 2)  # (B, T, C)

        # Apply vanilla transformer attention
        attn_output, _ = self.attention(q, k, v)  # (B, T, C)

        # Reshape back to (B, C, T)
        attn_output = attn_output.transpose(1, 2)  # (B, C, T)

        # Apply final projection
        output = self.out_projection(attn_output)

        return output


# ========================================================================
# Helper Building Blocks (Unchanged)
# ========================================================================
class ConvBNReLU(nn.Module):
    def __init__(self, c_in, c_out, k=7, s=1, p=None, groups=1):
        super().__init__()
        if p is None: p = k // 2
        self.conv = nn.Conv1d(c_in, c_out, k, s, p, groups=groups, bias=False)
        self.bn = nn.BatchNorm1d(c_out)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResBlock(nn.Module):
    def __init__(self, c, k=7):
        super().__init__()
        self.conv1 = ConvBNReLU(c, c, k)
        self.conv2 = nn.Conv1d(c, c, k, padding=k//2, bias=False)
        self.bn2 = nn.BatchNorm1d(c)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        identity = x
        y = self.conv1(x)
        return self.act(identity + y)


# ========================================================================
# Main U-Net Model
# ========================================================================
class UNet1D(nn.Module):
    """
    U-Net 1D with integrated Vanilla Transformer Attention blocks.
    """
    def __init__(self, in_ch=4, base=32, depth=4, n_heads=8, fs=200, bpm_range=(110, 170),
                 attn_loc="both"):
        super().__init__()
        self.depth = depth
        self.attn_loc = attn_loc
        chs = [base * (2**i) for i in range(depth)]

        # --- ENCODER ---
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.encoders.append(nn.Sequential(ConvBNReLU(in_ch, chs[0]), ResBlock(chs[0])))
        for i in range(depth - 1):
            self.pools.append(nn.MaxPool1d(2))
            self.encoders.append(nn.Sequential(ConvBNReLU(chs[i], chs[i+1]), ResBlock(chs[i+1])))

        # --- VANILLA TRANSFORMER ATTENTION BLOCKS ---
        if self.attn_loc in ("encoder", "both"):
            self.attn_block_enc1 = SparseAttentionBlock(chs[0], nhead=n_heads, fs=fs, bpm_range=bpm_range)
        if self.attn_loc in ("bottleneck", "both"):
            self.attn_block_bottleneck = SparseAttentionBlock(chs[-1], nhead=n_heads, fs=fs, bpm_range=bpm_range)

        # --- DECODER ---
        self.decoders = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            self.upsamples.append(nn.ConvTranspose1d(chs[i], chs[i-1], kernel_size=2, stride=2))
            self.decoders.append(nn.Sequential(ConvBNReLU(chs[i-1]*2, chs[i-1]), ResBlock(chs[i-1])))

        self.head = nn.Conv1d(chs[0], 1, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # --- Encoder path ---
        for i in range(self.depth):
            x = self.encoders[i](x)
            if (self.attn_loc in ("encoder", "both")) and i == 0:
                x = x + self.attn_block_enc1(x)  # Residual connection
            if i < self.depth - 1:
                skip_connections.append(x)
                x = self.pools[i](x)

        # --- Bottleneck attention ---
        if self.attn_loc in ("bottleneck", "both"):
            x = x + self.attn_block_bottleneck(x)  # Residual connection

        # --- Decoder path ---
        for i in range(self.depth - 1):
            x = self.upsamples[i](x)
            skip = skip_connections[-(i+1)]
            if x.shape[-1] != skip.shape[-1]:
                x = F.interpolate(x, size=skip.shape[-1], mode='linear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = self.decoders[i](x)

        return self.head(x)


# ========================================================================
# Smoke Test
# ========================================================================
if __name__ == "__main__":
    FS = 200  # Match the default fs in your notebook's config
    BPM_RANGE = (110, 170)
    model = UNet1D(
        in_ch=4,
        base=32,
        depth=4,
        n_heads=8,
        fs=FS,
        bpm_range=BPM_RANGE,
        attn_loc="both"
    ).to("cpu")

    x = torch.randn(2, 4, 4096)
    with torch.no_grad():
        y = model(x)
    print("Input shape:", x.shape)
    print("Output shape:", y.shape)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {num_params:,}")


In [ ]:
# Cell 5: Loss Functions and Performance Metrics

def estimate_pos_weight(loader: DataLoader, max_batches=50) -> torch.Tensor:
    """
    Estimates the positive class weight for BCEWithLogitsLoss by iterating
    through a few batches of a DataLoader.
    """
    pos_count, total_count = 0, 0
    for i, (_, y_fetal, _) in enumerate(loader):
        if i >= max_batches: break
        pos_count += y_fetal.sum().item()
        total_count += y_fetal.numel()
    
    neg_count = total_count - pos_count
    pos_weight = neg_count / (pos_count + 1e-6)
    return torch.tensor([pos_weight], device=DEVICE)

class FocalLoss(nn.Module):     #Addresses class imbalance (far more negative samples than peak samples)
    """
    Focal Loss, designed to address class imbalance.
    """
    def __init__(self, alpha=0.5, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        if self.pos_weight is not None:
            bce_loss = bce_loss * (targets * (self.pos_weight - 1) + 1)
        
        # # Focal Loss scaling 
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)    #probabilities of the true class
        focal_weight = (1 - pt).pow(self.gamma)
        # Alpha balancing
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        
        loss = alpha_t * focal_weight * bce_loss
        return loss.mean()

# --- Peak Detection and Metric Calculation ---

def pick_peaks(prob_map: np.ndarray, threshold: float, min_dist: int) -> np.ndarray:
    """Finds peaks in a probability map above a threshold."""
    peaks, _ = find_peaks(prob_map, height=threshold, distance=min_dist)
    return peaks.astype(np.int64)

def dense_to_peaks(y_mask: np.ndarray) -> np.ndarray:
    """
    Converts a dense binary mask back to single peak locations.
    """
    if y_mask.sum() == 0:
        return np.array([], dtype=np.int64)
    
    diff = np.diff(np.concatenate(([0], y_mask, [0])))
    starts = np.where(diff > 0)[0]
    ends = np.where(diff < 0)[0]
    
    centers = starts + (ends - starts) // 2
    return centers.astype(np.int64)

def match_peaks(pred_peaks: np.ndarray, true_peaks: np.ndarray, tolerance: int) -> tuple[int, int, int]:
    """
    Matches predicted peaks to true peaks within a tolerance window.
    Returns: (true positives, false positives, false negatives)
    """
    if true_peaks.size == 0 and pred_peaks.size == 0: return 0, 0, 0
    if true_peaks.size == 0: return 0, len(pred_peaks), 0
    if pred_peaks.size == 0: return 0, 0, len(true_peaks)
        
    true_peaks_sorted = np.sort(true_peaks)
    pred_peaks_sorted = np.sort(pred_peaks)
    
    matches = np.zeros_like(true_peaks, dtype=bool)
    tp = 0
    
    for pred in pred_peaks_sorted:
        distances = np.abs(true_peaks_sorted - pred)
        closest_idx = np.argmin(distances)
        
        if distances[closest_idx] <= tolerance and not matches[closest_idx]:
            tp += 1
            matches[closest_idx] = True
            
    fp = len(pred_peaks) - tp
    fn = len(true_peaks) - tp
    return tp, fp, fn

def compute_metrics_batch(logits: torch.Tensor, y_fetal: torch.Tensor, threshold: float) -> tuple:
    """Computes precision, recall, and F1 for a batch using corrected logic."""
    probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0, :]
    y = y_fetal.detach().cpu().numpy()[:, 0, :]
    
    tps, fps, fns = 0, 0, 0
    for i in range(probs.shape[0]):
        pred_p = pick_peaks(probs[i], threshold, MIN_PEAK_DIST)
        true_p = dense_to_peaks(y[i])
        tp, fp, fn = match_peaks(pred_p, true_p, EVAL_TOL_SAMPLES)
        tps += tp
        fps += fp
        fns += fn
        
    precision = tps / (tps + fps + 1e-9)
    recall = tps / (tps + fns + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1

# --- Smoke Test ---
print("\n--- Running a quick smoke test on the metric functions ---")
dummy_y = np.zeros(200); dummy_y[48:53] = 1.0; dummy_y[148:153] = 1.0
true_peaks_test = dense_to_peaks(dummy_y)
print(f"Dense mask converted to peaks: {true_peaks_test} (Expected: [50 150])")

pred_peaks_test = np.array([52, 100, 151])
tp, fp, fn = match_peaks(pred_peaks_test, true_peaks_test, tolerance=5)
print(f"Matched peaks: TP={tp}, FP={fp}, FN={fn} (Expected: TP=2, FP=1, FN=0)")
